In [1]:
import pandas as pd
import os

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

# Key in dataset title

In [2]:
# Load reference dataset-title lookup
ref = pd.read_parquet('Datasets/datasets_GBIF_OBIS.parquet', columns=['alternateIdentifier', 'title'])

# Build a unique lookup table
ref = ref.dropna(subset=['alternateIdentifier']).drop_duplicates(subset=['alternateIdentifier'], keep='first')
title_lookup = ref.set_index('alternateIdentifier')['title']

# Fill missing/empty datasetTitle values from the lookup
df = pd.read_parquet('MoF/merged_all_mof.parquet')
title_series = df['datasetTitle']
missing_title_mask = title_series.isna() | title_series.astype('string').str.strip().eq('')
mapped_titles = df.loc[missing_title_mask, 'datasetKey'].map(title_lookup)

filled_count = mapped_titles.notna().sum()
df.loc[missing_title_mask, 'datasetTitle'] = mapped_titles

# Save back to the same file
output_path = 'MoF/merged_all_mof.parquet'
df.to_parquet(output_path, index=False)

# Coverage summary
overall_coverage = (df['datasetTitle'].notna().sum() / len(df) * 100).round(2)

source_summary = (
    df.assign(_has_datasetTitle=df['datasetTitle'].notna())
      .groupby('sourceDatabase', dropna=False)
      .agg(
          total_records=('_has_datasetTitle', 'size'),
          datasetTitle_non_null=('_has_datasetTitle', 'sum')
      )
)
source_summary['datasetTitle_coverage_percent'] = (
    source_summary['datasetTitle_non_null'] / source_summary['total_records'] * 100
).round(2)

print(f"Filled datasetTitle for {filled_count:,} rows")
print(f"Overall datasetTitle coverage: {overall_coverage:.2f}%")
print("\nCoverage by sourceDatabase:")
print(source_summary.reset_index().to_string(index=False))
print(f"\nSaved to: {output_path}")

Filled datasetTitle for 2,573,327 rows
Overall datasetTitle coverage: 92.93%

Coverage by sourceDatabase:
sourceDatabase  total_records  datasetTitle_non_null  datasetTitle_coverage_percent
            AU          23858                      0                           0.00
   CRITTERBASE         240561                 240161                          99.83
          GBIF        2500130                2241015                          89.64
          OBIS        1339188                1339188                         100.00
          OGSL           7236                      0                           0.00

Saved to: MoF/merged_all_mof.parquet


# Check column coverage

In [3]:
# Load the parquet file
df = pd.read_parquet('MoF/merged_all_mof.parquet')

# Compute column coverage and dtypes overview
total_rows = len(df)
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null_count': df.notna().sum().values,
})
summary['coverage_percent'] = (summary['non_null_count'] / total_rows * 100).round(2)

# Save to Deduplicate folder
output_dir = 'Deduplicate'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'column_coverage.csv')
summary.to_csv(output_path, index=False)

# Optional: display the overview
print(summary)
print(f'\nSaved to: {output_path}')

                    column    dtype  non_null_count  coverage_percent
0   acceptedScientificName   string         4110954            100.00
1         acceptedTaxonKey    Int64         2500130             60.82
2                  aphiaID    Int64         4110973            100.00
3              areaSampled  float64          400256              9.74
4          areaSampledUnit   string          479472             11.66
..                     ...      ...             ...               ...
65      trawlingDistance_m  Float64            3752              0.09
66                    Keep      str         3580203             87.09
67         acceptedAphiaID  float64         4110973            100.00
68                isMarine  boolean         4109468             99.96
69              isBrackish  boolean         3311966             80.56

[70 rows x 4 columns]

Saved to: Deduplicate\column_coverage.csv


# Deduplicate

In [5]:
# Define deduplication columns
dedup_cols = ['acceptedAphiaID', 'decimalLatitudeStart', 'decimalLongitudeStart', 'year', 'startDayOfYear']

# Define prioritization columns in order
priority_cols = ['areaSampled', 'biomass', 'gear', 'depthA_m', 'individualCount']

# Count records before deduplication
records_before = len(df)
print(f"Records before deduplication: {records_before:,}")

# Create a sorting key: prioritize records with non-null values in priority columns
# Sort by: not null in priority_cols (False comes before True, so we negate), then keep first
df = pd.read_parquet('MoF/merged_all_mof.parquet')
df_sorted = df.copy()
for col in priority_cols:
	df_sorted[f'{col}_has_value'] = df_sorted[col].notna()

# Sort to bring records with values to the top
sort_cols = [f'{col}_has_value' for col in priority_cols]
df_sorted = df_sorted.sort_values(by=sort_cols, ascending=False)

# Drop duplicates keeping first (which now has priority)
df_deduplicated = df_sorted.drop_duplicates(subset=dedup_cols, keep='first')

# Remove temporary columns
df_deduplicated = df_deduplicated.drop(columns=sort_cols)

# Count records after deduplication
records_after = len(df_deduplicated)
records_removed = records_before - records_after

print(f"Records after deduplication: {records_after:,}")
print(f"Records removed: {records_removed:,}")
print(f"Removal rate: {(records_removed/records_before)*100:.2f}%\n")

# Count removals by sourceDatabase
removal_by_source = df.groupby('sourceDatabase').size() - df_deduplicated.groupby('sourceDatabase').size()
removal_stats = pd.DataFrame({
	'source': removal_by_source.index,
	'removed': removal_by_source.values,
	'total_before': df.groupby('sourceDatabase').size().values,
})
removal_stats['removal_percent'] = (removal_stats['removed'] / removal_stats['total_before'] * 100).round(2)
removal_stats = removal_stats.sort_values('removed', ascending=False)

print("Removals by sourceDatabase:")
print(removal_stats.to_string(index=False))

# Save deduplicated dataset
output_dir = 'Deduplicate'
output_path_dedup = os.path.join(output_dir, 'merged_all_deduplicated.parquet')
df_deduplicated.to_parquet(output_path_dedup)
print(f'\nSaved deduplicated data to: {output_path_dedup}')

Records before deduplication: 4,110,973
Records after deduplication: 2,486,130
Records removed: 1,624,843
Removal rate: 39.52%

Removals by sourceDatabase:
     source  removed  total_before  removal_percent
       OBIS   812942       1339188            60.70
       GBIF   737047       2500130            29.48
CRITTERBASE    61870        240561            25.72
         AU    12981         23858            54.41
       OGSL        3          7236             0.04

Saved deduplicated data to: Deduplicate\merged_all_deduplicated.parquet


In [7]:
# Read existing deduplicated file
df = pd.read_parquet(output_path_dedup)

# New deduplication keys (add datasetTitle)
dedup_subset = ['acceptedAphiaID', 'decimalLatitudeStart', 'decimalLongitudeStart', 'year', 'datasetTitle']

# Prioritization columns already defined as priority_cols
# Count before
records_before = len(df)
print(f"Records before deduplication: {records_before:,}")

# Create flags and sort to prioritise records with non-null priority cols
df_sorted = df.copy()
for col in priority_cols:
    df_sorted[f'{col}_has_value'] = df_sorted[col].notna()

sort_cols = [f'{c}_has_value' for c in priority_cols]
df_sorted = df_sorted.sort_values(by=sort_cols, ascending=False)

# Drop duplicates keeping the first (which has priority)
df_deduplicated2 = df_sorted.drop_duplicates(subset=dedup_subset, keep='first').drop(columns=sort_cols)

# Count after and report
records_after = len(df_deduplicated2)
records_removed = records_before - records_after
print(f"Records after deduplication: {records_after:,}")
print(f"Records removed: {records_removed:,}")
print(f"Removal rate: {(records_removed/records_before)*100:.2f}%\n")

# Removal stats by sourceDatabase
before_counts = df.groupby('sourceDatabase').size()
after_counts = df_deduplicated2.groupby('sourceDatabase').size()
all_sources = before_counts.index.union(after_counts.index)

before_counts = before_counts.reindex(all_sources, fill_value=0)
after_counts = after_counts.reindex(all_sources, fill_value=0)

removed_series = before_counts - after_counts
removal_stats = pd.DataFrame({
    'source': all_sources,
    'removed': removed_series.values,
    'total_before': before_counts.values
})
removal_stats['removal_percent'] = (removal_stats['removed'] / removal_stats['total_before'] * 100).replace([pd.NaT, pd.NA], 0).fillna(0).round(2)
removal_stats = removal_stats.sort_values('removed', ascending=False).reset_index(drop=True)

print("Removals by sourceDatabase:")
print(removal_stats.to_string(index=False))

# Overwrite the same parquet
df_deduplicated2.to_parquet(output_path_dedup, index=False)
print(f"\nSaved deduplicated data to: {output_path_dedup}")

Records before deduplication: 2,486,130
Records after deduplication: 2,445,527
Records removed: 40,603
Removal rate: 1.63%

Removals by sourceDatabase:
     source  removed  total_before  removal_percent
       GBIF    26554       1763083             1.51
       OBIS    12136        526246             2.31
CRITTERBASE     1862        178691             1.04
         AU       51         10877             0.47
       OGSL        0          7233             0.00

Saved deduplicated data to: Deduplicate\merged_all_deduplicated.parquet


# Check column coverage

In [8]:
# Load the parquet file
df = pd.read_parquet('Deduplicate/merged_all_deduplicated.parquet')

# Compute column coverage and dtypes overview
total_rows = len(df)
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'non_null_count': df.notna().sum().values,
})
summary['coverage_percent'] = (summary['non_null_count'] / total_rows * 100).round(2)

# Save to Deduplicate folder
output_dir = 'Deduplicate'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'column_coverage_deduplicated.csv')
summary.to_csv(output_path, index=False)

# Optional: display the overview
print(summary)
print(f'\nSaved to: {output_path}')

                    column    dtype  non_null_count  coverage_percent
0   acceptedScientificName   string         2445510            100.00
1         acceptedTaxonKey    Int64         1736529             71.01
2                  aphiaID    Int64         2445527            100.00
3              areaSampled  float64          321917             13.16
4          areaSampledUnit   string          325467             13.31
..                     ...      ...             ...               ...
65      trawlingDistance_m  Float64            3203              0.13
66                    Keep      str         2090562             85.49
67         acceptedAphiaID  float64         2445527            100.00
68                isMarine  boolean         2444519             99.96
69              isBrackish  boolean         1997374             81.67

[70 rows x 4 columns]

Saved to: Deduplicate\column_coverage_deduplicated.csv
